# From nodal elements to hierarchical high order

Python controls the experiment, while C++ implements the basis functions, orientation and global numbering. We start with $P_1$ triangles, $Q_1$ quadrilaterals and $P_1$ tetrahedra, then compare nodal $P_2$ coordinates with hierarchical modes. `HighOrderFESpace` supports every uniform positive degree in two dimensions.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ngsolve import *
from ngsolve.webgui import Draw
from ipywidgets import interactive, SelectionSlider
from IPython.display import display
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element2D, Element1D
from netgen.occ import unit_cube
from teaching_utils import load_module, square_mesh

m = load_module()

## Reference coordinates and local ordering

NGSolve orders triangle vertices as $(1,0),(0,1),(0,0)$, giving the basis $(\xi,\eta,1-\xi-\eta)$. The tetrahedron uses $(1,0,0),(0,1,0),(0,0,1),(0,0,0)$. Quadrilateral vertices are ordered cyclically as $(0,0),(1,0),(1,1),(0,1)$.

Local basis indices and the array returned by `GetDofNrs` must use the same ordering. The reference vertices are listed explicitly below. The module evaluates the basis values and gradients using the actual finite element classes.

In [ ]:
reference_vertices = {
    "triangle": [[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]],
    "quad": [[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]],
    "tetrahedron": [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0],
                    [0.0, 0.0, 1.0], [0.0, 0.0, 0.0]],
}
for kind, nodes in reference_vertices.items():
    print(f"{kind}: {len(nodes)} local degrees of freedom; nodes = {nodes}")

## Global vertex functions on triangular and quadrilateral meshes

For degree one, every global coefficient is attached to a mesh vertex. We set the coefficient at the central vertex to one and the others to zero. A triangular mesh produces a piecewise affine hat, while a quadrilateral mesh produces a mapped bilinear hat.

Both meshes below have the same vertices. Their local functions and cell connectivities differ. The WebGUI displays the actual finite element functions with the same height scale and color range. Inspect the element edges to see where each polynomial piece is defined.

The triangles use `P1TriangleFESpace`, whose element has explicit values and gradients. The quadrilaterals use `HighOrderFESpace` at degree one. In both spaces the vertex coefficients have the same numbers as the mesh vertices.

In [ ]:
global_examples = []
for quad in [False, True]:
    mesh = square_mesh(4, quadrilateral=quad)
    if quad:
        fes = m.HighOrderFESpace(mesh, order=1)
        label = "Q1 quadrilaterals"
    else:
        fes = m.P1TriangleFESpace(mesh)
        label = "P1 triangles"

    coordinates = np.array([vertex.point[:2] for vertex in mesh.vertices])
    center = np.argmin(np.linalg.norm(coordinates - [0.5, 0.5], axis=1))
    basis = GridFunction(fes)
    basis.vec[:] = 0
    basis.vec[int(center)] = 1
    print(label)
    Draw(basis, mesh, deformation=True, scale=0.5, min=0, max=1,
         euler_angles=[-60, 0, -35])
    global_examples.append({"quadrilateral": quad, "ndof": fes.ndof, "cells": mesh.ne})

## The same construction in three dimensions

On tetrahedra there are four affine reference functions and one global degree of freedom per vertex. The boundary elements are triangles.

Solve $-\Delta u+u=f$ with $u=g$ on the entire boundary, choosing $u=x+2y+3z$, hence $f=u$ and $g=u|_{\partial\Omega}$. This affine function belongs to the $P_1$ space. We check its reproduction by computing the $L^2$ error, which should be close to roundoff.

In [ ]:
mesh3 = Mesh(unit_cube.GenerateMesh(maxh=0.6))
fes3 = m.HighOrderFESpace(mesh3, order=1, dirichlet=".*")
u, v = fes3.TnT()
exact3 = x + 2 * y + 3 * z
a3 = BilinearForm((grad(u) * grad(v) + u * v) * dx)
a3.Assemble()
f3 = LinearForm(exact3 * v * dx)
f3.Assemble()
uh3 = GridFunction(fes3)
uh3.Set(exact3, definedon=mesh3.Boundaries(".*"))
rhs3 = f3.vec.CreateVector()
rhs3.data = f3.vec - a3.mat * uh3.vec
uh3.vec.data += a3.mat.Inverse(fes3.FreeDofs(), inverse="sparsecholesky") * rhs3

l2_error3 = sqrt(Integrate((uh3 - exact3)**2, mesh3, order=4))
print(f"L2 error: {l2_error3:.3e}")

`HighOrderFESpace` supplies the global numbering and the edge orientation used by the reference elements. In three dimensions this teaching space deliberately remains $P_1$. In two dimensions we now enrich the degree while retaining the earlier basis functions.

## Same quadratic space, different coordinates

The nodal $P_2$ basis is dual to values at vertices and edge midpoints. The hierarchical basis retains the three linear vertex functions and adds edge bubbles. These bases span the same polynomial space.

The manuscript first uses $E_{ij}=4\lambda_i\lambda_j$, with coefficient $U_{ij}=u(m_{ij})-[u(x_i)+u(x_j)]/2$. The implementation uses the integrated Legendre normalization

$$
\begin{align*}
\widetilde E_{ij}=-2\lambda_i\lambda_j=-\tfrac12 E_{ij},\qquad \widetilde U_{ij}=-2U_{ij}.
\end{align*}
$$

The vertex coefficients are unchanged. The last three entries of `hierarchical_coefficients` below therefore equal $-2$ times the manuscript's midpoint corrections, compared in the implementation's edge order. Both coefficient vectors represent the same polynomial.

We illustrate this with $u(\xi,\eta)=\xi^2+\eta$. The nodal coefficients are its values at the six nodes. In `hierarchical_at_nodes`, row $i$ corresponds to node $\widehat z_i$ and column $j$ to hierarchical basis function $\widehat\phi_j$:

$$
\begin{align*}
H_{ij}=\widehat\phi_j(\widehat z_i),\qquad Hc=a,
\end{align*}
$$

where $a$ contains the nodal values and $c$ the hierarchical coefficients. Solving this small system changes the coordinates of the same polynomial. This dense calculation illustrates the quadratic comparison; the high-order basis itself is constructed directly.

In [ ]:
# Vertices, then midpoints of edges (2,0), (1,2), (0,1).
nodes_p2 = [
    [1.0, 0.0], [0.0, 1.0], [0.0, 0.0],
    [0.5, 0.0], [0.0, 0.5], [0.5, 0.5],
]
nodal_coefficients = np.array([xi**2 + eta for xi, eta in nodes_p2])
hierarchical_at_nodes = np.array([
    m.HighOrderShapes("triangle", 2, point)[0] for point in nodes_p2
])
hierarchical_coefficients = np.linalg.solve(hierarchical_at_nodes, nodal_coefficients)
print(f"Nodal coefficients: {nodal_coefficients}")
print(f"Evaluation matrix H:\n{hierarchical_at_nodes}")
print(f"Hierarchical coefficients: {np.round(hierarchical_coefficients, 6)}")

## Integrated Legendre edges and Dubiner bubbles

For $n\ge2$, $L_n(t)=\int_{-1}^t P_{n-1}(s)\,ds$ vanishes at both endpoints. On a triangle, homogeneous scaling extends this edge function polynomially into the cell. Interior functions are $\lambda_0\lambda_1\lambda_2 D_{ij}$, where $D_{ij}$ is the Dubiner polynomial from the chapter. Multiplication by the bubble changes the orthogonality weight, so the resulting interior modes need not be mutually $L^2$-orthogonal.

Explore six basis functions on a single $P_4$ reference triangle: one vertex function, three modes on the same oriented edge (degrees $n=2,3,4$), and the interior modes $bD_{00}=b$ and $bD_{01}=b(3\lambda_2-1)$, with $b=\lambda_0\lambda_1\lambda_2$. Setting one coefficient to one and all others to zero selects the corresponding basis function.

The **Basis** slider selects the function. Compare their boundary traces, the even/odd symmetry along the edge, and the interior sign change. The height scale is adjusted for each mode so that small bubbles remain visible; heights across modes therefore do not directly compare amplitudes. The colorbar shows the unscaled function values.

Increasing the degree retains these functions, although their positions in the coefficient vector can change when the edge blocks grow.

In [ ]:
triangle_netgen = NetgenMesh(dim=2)
triangle_domain = triangle_netgen.AddRegion("domain", dim=2)
triangle_boundary = triangle_netgen.AddRegion("boundary", dim=1)
triangle_points = [triangle_netgen.Add(MeshPoint(Pnt(a, b, 0)))
                   for a, b in reference_vertices["triangle"]]
triangle_netgen.Add(Element2D(triangle_domain, triangle_points))
for a, b in [(0, 1), (1, 2), (2, 0)]:
    triangle_netgen.Add(Element1D([triangle_points[a], triangle_points[b]],
                                index=triangle_boundary))
triangle_mesh = Mesh(triangle_netgen)

triangle_order = 4
triangle_fes = m.HighOrderFESpace(triangle_mesh, order=triangle_order)

triangle_cell = list(triangle_mesh.Elements())[0]
triangle_dofs = list(triangle_fes.GetDofNrs(triangle_cell))
triangle_first_interior = 3 + 3 * (triangle_order - 1)

triangle_modes = [0, 3, 4, 5, triangle_first_interior, triangle_first_interior + 1]
triangle_scales = [0.5, 1, 2.5, 4, 12, 30]
triangle_basis = GridFunction(triangle_fes)

def draw_triangle_mode(mode):
    triangle_basis.vec[:] = 0
    triangle_basis.vec[triangle_dofs[triangle_modes[mode]]] = 1
    return Draw(triangle_basis, triangle_mesh, deformation=True,
                order=8, scale=triangle_scales[mode], euler_angles=[-60, 0, -35])

triangle_selector = SelectionSlider(
    options=[("Vertex", 0), ("Edge n=2", 1), ("Edge n=3", 2), ("Edge n=4", 3),
             ("Bubble b", 4), ("Bubble b D01", 5)],
    description="Basis", continuous_update=True)
triangle_view = interactive(draw_triangle_mode, mode=triangle_selector)
display(triangle_view)

## Shared edge, shared coefficients

Use two triangles and degree three. Each edge has an even mode of degree two and an odd mode of degree three. The global edge direction runs from the smaller to the larger vertex number. The local-to-global map identifies the same two edge coefficients in both cells. The reference element incorporates the orientation sign into its shapes.

We set only the shared cubic edge coefficient to one. Compare the two traces at the same physical edge points. Removing the orientation sign reverses one trace. For nodal cubic elements, the corresponding operation exchanges two edge nodes instead.

The C++ space supplies global vertex numbers through `SetVertexNumbers`. NGSolve's base class `VertexOrientedFE<ET>` supplies the ordered endpoints through `GetVertexOrientedEdge`, and the basis formulas use these endpoints directly.

In [ ]:
patch = square_mesh(1)
patch_fes = m.HighOrderFESpace(patch, order=3)
patch_cells = list(patch.Elements())
second_edges = [edge.nr for edge in patch_cells[1].edges]
shared_edges = [edge.nr for edge in patch_cells[0].edges if edge.nr in second_edges]
shared_edge = shared_edges[0]
second_vertices = [vertex.nr for vertex in patch_cells[1].vertices]
shared_vertices = sorted([
    vertex.nr for vertex in patch_cells[0].vertices if vertex.nr in second_vertices
])

reference_triangle = np.array(reference_vertices["triangle"])
reference_edges = [(2, 0), (1, 2), (0, 1)]
edge_parameters = np.array([0.2, 0.4, 0.7])
oriented_traces = []
raw_traces = []
shared_edge_dofs = []
for cell in patch_cells:
    vertices = [vertex.nr for vertex in cell.vertices]
    edge = [edge.nr for edge in cell.edges].index(shared_edge)
    a, b = reference_edges[edge]
    dofs = list(patch_fes.GetDofNrs(cell))
    odd_mode = 3 + 2 * edge + 1
    shared_edge_dofs.append(dofs[odd_mode])
    print(f"cell {cell.nr}: vertices {vertices}, local edge {vertices[a], vertices[b]}")
    print("  local-to-global map:", dofs)
    i, j = [vertices.index(vertex) for vertex in shared_vertices]
    points_on_edge = [(1 - t) * reference_triangle[i] + t * reference_triangle[j]
                      for t in edge_parameters]
    trace = np.array([
        m.HighOrderShapes("triangle", 3, point.tolist(), vertices)[0][odd_mode]
        for point in points_on_edge
    ])
    orientation = 1 if vertices[a] < vertices[b] else -1
    oriented_traces.append(trace)
    raw_traces.append(orientation * trace)

print("Shared cubic coefficient numbers:", shared_edge_dofs)
print("Traces without orientation signs:", np.round(raw_traces, 6))
print("Traces with orientation signs:", np.round(oriented_traces, 6))
shared_trace_difference = np.max(np.abs(oriented_traces[0] - oriented_traces[1]))
print(f"Maximum trace difference: {shared_trace_difference:.3e}")

## A shared edge between a triangle and a quadrilateral

Now join a $P_4$ triangle to a $Q_4$ quadrilateral along the vertical edge $x=1$. Both elements use the same three global coefficients on this edge. For each mode, we set one of these coefficients to one and all other coefficients to zero.

The **Degree** slider switches between edge degrees 2, 3 and 4. Rotate the view to inspect the common edge: the function values agree, although the extensions into the two cells differ. Their gradients may jump. The cubic mode also shows why both cells must use the same edge orientation.

The height scale is adjusted for each mode; the colorbar gives the unscaled function values.

In [ ]:
mixed_netgen = NetgenMesh(dim=2)
mixed_domain = mixed_netgen.AddRegion("domain", dim=2)
mixed_boundary = mixed_netgen.AddRegion("boundary", dim=1)
mixed_coordinates = [(0, 0), (1, 0), (1, 1), (2, 0), (2, 1)]
mixed_points = [mixed_netgen.Add(MeshPoint(Pnt(a, b, 0)))
                for a, b in mixed_coordinates]
for vertices in [(0, 1, 2), (1, 3, 4, 2)]:
    mixed_netgen.Add(Element2D(mixed_domain, [mixed_points[i] for i in vertices]))
for a, b in [(0, 1), (1, 3), (3, 4), (4, 2), (2, 0)]:
    mixed_netgen.Add(Element1D([mixed_points[a], mixed_points[b]],
                              index=mixed_boundary))

mixed_mesh = Mesh(mixed_netgen)
mixed_fes = m.HighOrderFESpace(mixed_mesh, order=4)
mixed_cells = list(mixed_mesh.Elements())
mixed_second_edges = [edge.nr for edge in mixed_cells[1].edges]
mixed_shared_edges = [edge.nr for edge in mixed_cells[0].edges
                      if edge.nr in mixed_second_edges]
mixed_shared_edge = mixed_shared_edges[0]
mixed_shared_dofs = list(mixed_fes.GetDofNrs(NodeId(EDGE, mixed_shared_edge)))
mixed_basis = GridFunction(mixed_fes)
mixed_scales = [1, 2.5, 4]

def draw_mixed_mode(mode):
    mixed_basis.vec[:] = 0
    mixed_basis.vec[mixed_shared_dofs[mode]] = 1
    return Draw(mixed_basis, mixed_mesh, deformation=True,
                order=8, scale=mixed_scales[mode], euler_angles=[-60, 0, -35])

print("Shared edge coefficients, in degree order 2, 3, 4:", mixed_shared_dofs)
mixed_selector = SelectionSlider(
    options=[("n=2", 0), ("n=3", 1), ("n=4", 2)], value=1,
    description="Degree", continuous_update=True)
mixed_view = interactive(draw_mixed_mode, mode=mixed_selector)
display(mixed_view)

## Increasing the degree on a fixed mesh

Keep the mesh fixed and solve $-\Delta u=f$ with

$$
\begin{align*}
u=e^{2x+y}\sin(\pi x)\sin(\pi y),\qquad f=-\Delta u,
\end{align*}
$$

and homogeneous Dirichlet data. The source is computed by differentiating the prescribed exact function. Triangles use $P_p$, quadrilaterals use mapped $Q_p$.

The WebGUI shows the exact target function. For each element type, the mesh remains fixed while the polynomial degree increases.

For each degree, compare our computed **function** with a separately assembled native `H1` solution. Their bases differ, so comparing coefficient arrays would not compare the represented functions. We instead compute their $L^2$ difference. Quadrature order is increased with the degree.

In [ ]:
from teaching_utils import errors

smooth = exp(2 * x + y) * sin(pi * x) * sin(pi * y)
smooth_source = -smooth.Diff(x).Diff(x) - smooth.Diff(y).Diff(y)
preview_mesh = square_mesh(2)
Draw(smooth, preview_mesh)

In [ ]:
p_convergence = []
for quad in [False, True]:
    mesh_p = square_mesh(2, quadrilateral=quad)
    for degree in range(1, 7):
        space_p = m.HighOrderFESpace(mesh_p, order=degree, dirichlet=".*")
        quadrature_order = 2 * degree + 10
        mat_p = m.AssembleMatrix(space_p, [m.Laplace(CF(1), quadrature_order=quadrature_order)])
        rhs_p = m.AssembleVector(space_p, [m.Source(smooth_source, quadrature_order=quadrature_order)])
        own_p = GridFunction(space_p)
        own_p.vec.data = mat_p.Inverse(space_p.FreeDofs(), inverse="sparsecholesky") * rhs_p

        native_p = H1(mesh_p, order=degree, dirichlet=".*")
        u, v = native_p.TnT()
        a_native = BilinearForm(grad(u) * grad(v) * dx(bonus_intorder=quadrature_order))
        a_native.Assemble()
        f_native = LinearForm(smooth_source * v * dx(bonus_intorder=quadrature_order))
        f_native.Assemble()
        field_p = GridFunction(native_p)
        field_p.vec.data = a_native.mat.Inverse(native_p.FreeDofs(), inverse="sparsecholesky") * f_native.vec

        error = errors(mesh_p, own_p, smooth, quadrature_order=quadrature_order)
        native_difference = sqrt(Integrate((own_p - field_p)**2, mesh_p, order=quadrature_order))
        p_convergence.append({
            "quad": quad,
            "p": degree,
            "ndof": space_p.ndof,
            "native_difference": native_difference,
            "L2": error["L2"],
            "H1": error["H1"],
        })

Plot the $L^2$ and $H^1$ errors against the degree. Decreasing errors show the effect of enriching the space without refining the mesh. The separate comparison with native `H1` checks that our basis and assembly produce the same discrete function.

In [ ]:
plt.figure(figsize=(7, 4))
for quad in [False, True]:
    rows = [row for row in p_convergence if row["quad"] == quad]
    degrees = [row["p"] for row in rows]
    element_label = "Qp" if quad else "Pp"
    for norm, marker in [("H1", "o-"), ("L2", "s--")]:
        error_values = [row[norm] for row in rows]
        plt.semilogy(degrees, error_values, marker, label=f"{norm}, {element_label}")
plt.xlabel("uniform degree p on a fixed mesh")
plt.ylabel("error")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Largest L2 difference to native H1:", max(row["native_difference"] for row in p_convergence))